# 02 · Clasificación con árbol de decisión

**Responsable:** Ignacio González

**Pregunta:** ¿un día va a ser de alta demanda de viajes para una estación y un tipo de residuo?

**Datos:** `data/processed/datos_limpios.csv`, generado en `01_preparacion.ipynb`.

In [1]:
# Paso 1 · Librerías

import pandas as pd
import matplotlib.pyplot as plt

# scikit-learn: la librería de aprendizaje automático.
# DecisionTreeClassifier es el árbol de decisión para clasificar;
# plot_tree lo dibuja (lo usamos al final para leer sus reglas)
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Métricas para evaluar el modelo (las usamos en la Etapa 4)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

In [2]:
# Paso 2 · Cargar los datos limpios

# parse_dates convierte "fecha" a tipo fecha al leer (en el CSV se guarda como texto)
df = pd.read_csv("../data/processed/datos_limpios.csv", parse_dates=["fecha"])

print(df.shape)
df.head()

(12408, 11)


,fecha,estacion,tipo,toneladas,viajes,anio,mes,dia_semana,feriado,toneladas_sem_ant,viajes_sem_ant
0,2017-01-09,Norte,Basura,99.340123,30,2017,1,Lunes,0,70.523624,23
1,2017-01-09,Norte,Organicos,27.591310,7,2017,1,Lunes,0,23.750000,7
2,2017-01-09,Sur,Basura,325.610668,99,2017,1,Lunes,0,364.491521,111
3,2017-01-09,Sur,Organicos,211.350862,49,2017,1,Lunes,0,233.762208,48
4,2017-01-09,Sur,Reciclables,402.682309,85,2017,1,Lunes,0,402.273225,93


In [3]:
# Paso 3 · Variables de entrada

# Qué hacemos: definir qué columnas puede usar el modelo para predecir.

# Por qué: es acá donde se aplica la regla contra la fuga de información. 
# Si se escribe la lista explícitamente
# queda claro qué se usa y qué no, y nadie puede mandar viajes por error.

# Lista explícita de entradas permitidas.
# NO están 'viajes' ni 'toneladas' del mismo día: 'viajes' define la categoría
# que queremos predecir, y 'toneladas' está casi perfectamente relacionada con ella
# (correlación de 0,95). Usarlas sería darle la respuesta al modelo.
entradas = [
    "estacion", "tipo", "dia_semana",      # categóricas (texto)
    "anio", "mes", "feriado",              # de calendario (números)
    "toneladas_sem_ant", "viajes_sem_ant", # semana anterior: ya ocurrieron, son válidas
]

df[entradas].head()

,estacion,tipo,dia_semana,anio,mes,feriado,toneladas_sem_ant,viajes_sem_ant
0,Norte,Basura,Lunes,2017,1,0,70.523624,23
1,Norte,Organicos,Lunes,2017,1,0,23.750000,7
2,Sur,Basura,Lunes,2017,1,0,364.491521,111
3,Sur,Organicos,Lunes,2017,1,0,233.762208,48
4,Sur,Reciclables,Lunes,2017,1,0,402.273225,93


Variables de entrada: estación, tipo de residuo, día de la semana, año, mes, feriado y los valores de la semana anterior. Se excluyen viajes y toneladas del mismo día porque revelarían la respuesta: la categoría se construye con los viajes, y las toneladas tienen una correlación de 0,95 con ellos

In [ ]:
# Paso 4 · Codificación one-hot

# Qué hacemos: transformar las columnas de texto (estación, tipo, día) en columnas de 0 y 1.

# Por qué: scikit-learn solo trabaja con números. 
# No podemos reemplazar “Norte” por 1 y “Sur” por 2, porque el modelo interpretaría que Sur es “más” que Norte, y eso no significa nada. 
# La solución se llama codificación one-hot: una columna por cada valor posible, con 1 si la fila tiene ese valor y 0 si no.

# get_dummies crea una columna de 0 y 1 por cada valor de las columnas indicadas.
# Las columnas numéricas (anio, mes, feriado, semana anterior) quedan como están.
# dtype=int hace que los valores sean 0 y 1 (si no, serían True y False)
X = pd.get_dummies(
    df[entradas],
    columns=["estacion", "tipo", "dia_semana"],
    dtype=int,
)

print(X.shape)
print(X.columns.tolist())
X.head()

(12408, 15)
['anio', 'mes', 'feriado', 'toneladas_sem_ant', 'viajes_sem_ant', 'estacion_Norte', 'estacion_Sur', 'tipo_Basura', 'tipo_Organicos', 'tipo_Reciclables', 'dia_semana_Jueves', 'dia_semana_Lunes', 'dia_semana_Martes', 'dia_semana_Miercoles', 'dia_semana_Viernes']


,anio,mes,feriado,toneladas_sem_ant,viajes_sem_ant,estacion_Norte,estacion_Sur,tipo_Basura,tipo_Organicos,tipo_Reciclables,dia_semana_Jueves,dia_semana_Lunes,dia_semana_Martes,dia_semana_Miercoles,dia_semana_Viernes
0,2017,1,0,70.523624,23,1,0,1,0,0,0,1,0,0,0
1,2017,1,0,23.750000,7,1,0,0,1,0,0,1,0,0,0
2,2017,1,0,364.491521,111,0,1,1,0,0,0,1,0,0,0
3,2017,1,0,233.762208,48,0,1,0,1,0,0,1,0,0,0
4,2017,1,0,402.273225,93,0,1,0,0,1,0,1,0,0,0


estacion	→	estacion_Norte	estacion_Sur
Norte		           1	         0
Sur		               0	         1

Codificación one-hot: las variables categóricas se transformaron en columnas de 0 y 1, una por cada valor posible. No se usaron números consecutivos (Norte = 1, Sur = 2) porque el modelo interpretaría un orden que no existe.